In [2]:
# --- PER-ITEM SENTIMENT & EMOTION FEATURES ---
# Logic: Score every unique item text with two transformer pipelines - a
#        3-class sentiment model and a 7-class emotion model - to attach
#        affective features to each item.
# Workflow: <split>_item_list.parquet -> per-item sentiment+emotion vectors ->
#           <split>_item_list_sentiment.parquet.

import os
import polars as pl
import torch
from transformers import pipeline

# ---- Config ----

# Which split to score: "" (training), "holdout_" or "validation_"
splits = ["validation_", "holdout_", ""]
for split in splits:

    batch_size   = 256
    parquet_path = f"../../data/raw/{split}item_list.parquet"
    output_path  = f"../../data/processed/{split}item_list_sentiment.parquet"

    os.makedirs("../../data/processed", exist_ok=True)

    # ---- Load items ----

    item_list = pl.read_parquet(parquet_path)
    items = item_list["item"].to_list()
    print(f"Processing {len(items)} items...")

    # ---- Pipelines (top_k=None returns the full probability vector) ----

    pipe_kwargs = {
        "device": 0,
        "batch_size": batch_size,
        "model_kwargs": {"torch_dtype": torch.float16},
    }

    print("Initializing pipelines...")
    sent_pipe = pipeline(
        "text-classification",
        model="cardiffnlp/twitter-roberta-base-sentiment-latest",
        top_k=None, **pipe_kwargs,
    )
    hart_pipe = pipeline(
        "text-classification",
        model="j-hartmann/emotion-english-distilroberta-base",
        top_k=None, **pipe_kwargs,
    )

    # ---- Score ----

    print("Running sentiment...")
    sent_res = sent_pipe(items, truncation=True, max_length=64)
    print("Running Hartmann emotion vector...")
    hart_res = hart_pipe(items, truncation=True, max_length=64)

    # ---- Unpack per-class probabilities ----

    # Sentiment: positive / neutral / negative
    sent_cols = {f"sent_{label}": [] for label in [r["label"] for r in sent_res[0]]}
    for res in sent_res:
        for r in res:
            sent_cols[f"sent_{r['label']}"].append(r["score"])

    # Hartmann: anger / disgust / fear / joy / neutral / sadness / surprise
    hart_cols = {f"emo_{label}": [] for label in [r["label"] for r in hart_res[0]]}
    for res in hart_res:
        for r in res:
            hart_cols[f"emo_{r['label']}"].append(r["score"])

    # ---- Assemble final table ----

    new_features = pl.DataFrame({**sent_cols, **hart_cols}).with_columns([
        # Top labels - handy for ad-hoc inspection but not used by the model
        pl.Series("top_sentiment", [r[0]["label"] for r in sent_res]),
        pl.Series("top_emotion",   [r[0]["label"] for r in hart_res]),
    ])

    final_df = pl.concat([item_list, new_features], how="horizontal")

    # ---- Save ----

    final_df.write_parquet(output_path)
    print(f"Success! Saved {len(final_df)} items to {output_path}")


Processing 262 items...
Initializing pipelines...


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Device set to use cuda:0


Running sentiment...
Running Hartmann emotion vector...
Success! Saved 262 items to ../../data/processed/validation_item_list_sentiment.parquet
Processing 418 items...
Initializing pipelines...


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Device set to use cuda:0


Running sentiment...
Running Hartmann emotion vector...
Success! Saved 418 items to ../../data/processed/holdout_item_list_sentiment.parquet
Processing 3127 items...
Initializing pipelines...


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Device set to use cuda:0


Running sentiment...
Running Hartmann emotion vector...
Success! Saved 3127 items to ../../data/processed/item_list_sentiment.parquet
